# Tweet Preprocessing and Sentiment Labelling

This notebook processes tweets from the tweets folder, cleans them, and applies FinRoBERTa sentiment analysis.

**Process:**
1. Load tweet CSV files from tweets directory
2. Clean and preprocess tweet text (handles mentions, hashtags, URLs, emojis)
3. Apply FinRoBERTa sentiment analysis
4. Save labelled tweets with sentiment scores

## 1. Import Required Libraries

Import libraries for data processing, sentiment analysis, and file handling.

In [2]:
import os
import re
from pathlib import Path
import pandas as pd
import torch
from lingua import LanguageDetectorBuilder, Language
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Define the lingua English detector
lingua_detector = LanguageDetectorBuilder.from_all_languages().with_minimum_relative_distance(0.25).build()
print("Language Classifier loaded.")

Language Classifier loaded.


## 2. Directories and Settings

Set input directory (raw tweets) and output directory (processed tweets with sentiment).

**Directory Structure:**
- Input: `Raw_Data/Tweets/` folder containing `tweets_TICKER.csv` files
- Output: `Processed_Data/tweets_labelled/` folder for processed results

In [3]:
# Directories definitions
BASE_DIR = Path.cwd().resolve()
if (BASE_DIR / "Raw_Data" / "Tweets").exists():
    pass
elif (BASE_DIR.parent / "Raw_Data" / "Tweets").exists():
    BASE_DIR = BASE_DIR.parent

INPUT_DIR = str(BASE_DIR / "Raw_Data" / "Tweets")
OUTPUT_DIR = BASE_DIR / "Processed_Data" / "tweets_sentiment_daily"

print(f"BASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP
INPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\Raw_Data\Tweets
OUTPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\Processed_Data\tweets_sentiment_daily


## 3. Load FIN-RoBERTa Model

FIN-RoBERTa is a custom RoBERTa model fine-tuned on financial text for sentiment analysis.

**Model Details:**
- Model: alasteirho/FIN-RoBERTa-Custom
- Output: 3 classes (positive, negative, neutral)
- Sentiment Score: P(positive) - P(negative), range [-1, 1]

In [4]:
# Load FinRoBERTa tokenizer and model for financial sentiment analysis
tokenizer = AutoTokenizer.from_pretrained('alasteirho/FIN-RoBERTa-Custom')
model = AutoModelForSequenceClassification.from_pretrained('alasteirho/FIN-RoBERTa-Custom')

# Set to evaluation mode (disables dropout)
model.eval()

# Detect available hardware accelerator
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
model.to(device)

print(f"Using device: {device}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Using device: cuda


## 4. Define Preprocessing and Filtering Functions

Data cleaning follows the methodology by Wilksch & Abramova (2023).

**DataFrame-Level Filtering (applied per ticker file):**
1. Remove hyperlinks (needed for accurate word counts in subsequent steps)
2. Drop duplicates (exact duplicates + duplicate texts longer than 5 words to catch bots)
3. Filter tweets with 5+ cashtags or 8+ hashtags (spam indication)
4. Remove spam by ratio filters (cashtag/word, hashtag/word, mention/word ratios must be ≤ 0.5)
5. Remove cryptocurrency posts (allow ≤ 2 crypto keywords per tweet)
6. Filter non-English tweets using lingua

**Text-Level Cleaning (for FinRoBERTa input):**
- Replace the $TICKER with company name (e.g. $AAPL to Apple)
- Remove all other $TICKER cashtags (Apart from the ticker thats mentioned in file name)
- Convert #hashtags to readable text
- Remove emojis, special characters, excessive punctuation
- Filter very short tweets (< 10 characters, noise)

In [ ]:
def is_english(text, min_alpha_chars=20):
    # Check if text is English using lingua.
    if not text or not isinstance(text, str):
        return False

    sample = re.sub(r'http\S+|www\.\S+', ' ', text)
    sample = re.sub(r'@\w+', ' ', sample)
    sample = re.sub(r'\$[^\d\s]{1,5}', ' ', sample)
    sample = sample.replace('#', ' ')
    sample = re.sub(r'[^A-Za-z\s]', ' ', sample)
    sample = re.sub(r'\s+', ' ', sample).strip()

    if not sample:
        return False

    alpha_chars = len(re.sub(r'[^A-Za-z]', '', sample))
    if alpha_chars < min_alpha_chars:
        return True

    detected = lingua_detector.detect_language_of(sample)
    return detected == Language.ENGLISH

# Map ticker to company name for replacement
TICKER_TO_NAME = {
    'AAPL': 'Apple', 'AMZN': 'Amazon', 'AVGO': 'Broadcom',
    'BRK.B': 'Berkshire Hathaway', 'GOOGL': 'Google', 'HD': 'Home Depot',
    'JNJ': 'Johnson and Johnson', 'JPM': 'JPMorgan', 'LLY': 'Eli Lilly',
    'MA': 'Mastercard', 'META': 'Meta', 'MSFT': 'Microsoft',
    'NVDA': 'Nvidia', 'ORCL': 'Oracle', 'PG': 'Procter and Gamble',
    'TSLA': 'Tesla', 'UNH': 'UnitedHealth', 'V': 'Visa',
    'WMT': 'Walmart', 'XOM': 'ExxonMobil',
}

# Cryptocurrency keywords for filtering (PyFin-sentiment: Towards a machine-learning-based model for deriving sentiment from financial tweets)
CRYPTO_KEYWORDS = {
    'bitcoin', 'etherium', 'btc', 'eth', 'nft', 'token', 'wallet',
    'web3', 'airdrop', 'wagmi', 'solana', 'opensea', 'cryptopunks',
    'uniswap', 'lunar', 'hodl', 'binance', 'coinbase', 'cryptocom',
    'doge',
}

def remove_hyperlinks(text):
    # Remove all URLs/hyperlinks from tweet text.
    if pd.isna(text) or not isinstance(text, str):
        return text
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'bit\.ly/\S+|goo\.gl/\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def count_cashtags(text):
    # Count number of $TICKER cashtags in text.
    if pd.isna(text) or not isinstance(text, str):
        return 0
    return len(re.findall(r'\$[A-Za-z]{1,5}\b', text))

def count_hashtags(text):
    # Count number of #hashtags in text.
    if pd.isna(text) or not isinstance(text, str):
        return 0
    return len(re.findall(r'#\w+', text))

def count_mentions(text):
    # Count number of @mentions in text.
    if pd.isna(text) or not isinstance(text, str):
        return 0
    return len(re.findall(r'@\w+', text))

def count_words(text):
    # Count words in text.
    if pd.isna(text) or not isinstance(text, str):
        return 0
    return len(text.split())

def count_crypto_keywords(text):
    # Count how many cryptocurrency keywords appear in text.
    if pd.isna(text) or not isinstance(text, str):
        return 0
    words = set(re.findall(r'[a-zA-Z]+', text.lower()))
    return len(words & CRYPTO_KEYWORDS)

def clean_for_model(tweet_text, ticker=None):
    # Clean tweet text for FinRoBERTa input. Applied after DataFrame-level filtering.
    if pd.isna(tweet_text) or not isinstance(tweet_text, str):
        return None

    # Remove newlines
    tweet_text = tweet_text.replace('\n', ' ').replace('\r', ' ')
    tweet_text = re.sub(r'\s+', ' ', tweet_text).strip()

    # URLs already removed in earlier step, but catch any remaining
    tweet_text = re.sub(r'http\S+|www\.\S+', '', tweet_text)

    # Remove Twitter mentions at the start of tweets (replies)
    tweet_text = re.sub(r'^(@\w+\s*)+', '', tweet_text)

    # Convert hashtags to readable text (#AppleStock -> Apple Stock)
    def hashtag_to_text(match):
        tag = match.group(1)
        spaced = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        return spaced
    tweet_text = re.sub(r'#(\w+)', hashtag_to_text, tweet_text)

    # Replace the file's own $TICKER with company name
    if ticker:
        company_name = TICKER_TO_NAME.get(ticker, ticker)
        escaped_ticker = re.escape(ticker)
        ticker_variants = [escaped_ticker]
        if '.' in ticker:
            ticker_variants.append(re.escape(ticker.split('.')[0]))
        own_pattern = r'\$(?:' + '|'.join(ticker_variants) + r')\b'
        tweet_text = re.sub(own_pattern, company_name, tweet_text)

    # Remove any remaining $TICKER mentions (other tickers)
    tweet_text = re.sub(r'\$[^\d\s]{1,5}', '', tweet_text)

    # Remove excessive punctuation (>3 repeated characters)
    tweet_text = re.sub(r'([!?.])\\1{2,}', r'\1', tweet_text)

    # Remove emojis and special characters
    tweet_text = re.sub(r'[^\w\s.,!?-]', '', tweet_text)

    # Final whitespace cleanup
    tweet_text = re.sub(r'\s+', ' ', tweet_text).strip()

    # Filter out very short tweets
    if len(tweet_text) < 10:
        return None

    return tweet_text

## 5. Define Sentiment Analysis Function

Apply FinRoBERTa model to get sentiment scores for cleaned tweets.

**Sentiment Calculation:**
- Gets probability distribution over [negative, neutral, positive]
- Sentiment score = P(positive) - P(negative)
- Range: -1 (very negative) to +1 (very positive)

In [6]:
def get_sentiment(text):
    # Handle empty or None text
    if not text or pd.isna(text):
        return None
    
    # Tokenize and prepare input for FinRoBERTa
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=512,
        padding=True
    )
    
    # Move input tensors to same device as model
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    # Get model predictions without computing gradients
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Extract probabilities: [negative, neutral, positive]
    probabilities = predictions[0].cpu().numpy()
    positive_prob = probabilities[2]
    negative_prob = probabilities[0]
    
    # Calculate sentiment score: positive - negative
    sentiment_score = positive_prob - negative_prob
    
    return sentiment_score

## 6. Get List of Tweet CSV Files

Find all tweet CSV files in the input directory (format: `tweets_TICKER.csv`).

In [7]:
# Get all CSV files from tweets directory
csv_files = [
    filename 
    for filename in os.listdir(INPUT_DIR) 
    if filename.endswith('.csv') and filename.startswith('tweets_')
]

print(f"Found {len(csv_files)} tweet files to process:")
for file in csv_files:
    print(f"  - {file}")

Found 20 tweet files to process:
  - tweets_AAPL.csv
  - tweets_AMZN.csv
  - tweets_AVGO.csv
  - tweets_BRK.B.csv
  - tweets_GOOGL.csv
  - tweets_HD.csv
  - tweets_JNJ.csv
  - tweets_JPM.csv
  - tweets_LLY.csv
  - tweets_MA.csv
  - tweets_META.csv
  - tweets_MSFT.csv
  - tweets_NVDA.csv
  - tweets_ORCL.csv
  - tweets_PG.csv
  - tweets_TSLA.csv
  - tweets_UNH.csv
  - tweets_V.csv
  - tweets_WMT.csv
  - tweets_XOM.csv


## 8. Process All Tweet Files

For each ticker's tweet file, apply the full cleaning pipeline (Wilksch & Abramova, 2023):

1. **Data collection** — Load raw tweets
2. **Drop duplicates** — Remove exact duplicates and bot-like repeated texts (>5 words)
3. **Filter cashtags & hashtags** — Remove tweets with ≥5 cashtags or ≥8 hashtags
4. **Remove spam by ratios** — Require cashtag/word, hashtag/word, mention/word ≤ 0.5
5. **Remove cryptocurrency posts** — Allow ≤ 2 crypto keywords per tweet
6. **Filter non-English** — Remove non-English tweets using lingua
7. **Clean text for model** — Prepare text for FinRoBERTa, replace own $TICKER with company name, remove other tickers, filter short tweets

A summary table is displayed per ticker showing tweet counts after each step.

In [8]:
# Dictionary used to store statistics across all files
global_statistic = {
    'Data collection': 0,
    'Drop duplicates': 0,
    'Filter number of cashtags & hashtags': 0,
    'Remove spam by ratios': 0,
    'Remove cryptocurrency posts': 0,
    'Filter non-English': 0,
    'Clean text for model': 0,
}

for csv_file in csv_files:
    print(f"\n{'='*70}")
    print(f"Processing {csv_file}")
    print(f"{'='*70}")
    
    # Extract ticker from filename (e.g. tweets_AAPL.csv -> AAPL)
    ticker = csv_file.replace('tweets_', '').replace('.csv', '')
    
    # Load tweet data
    input_path = os.path.join(INPUT_DIR, csv_file)
    df = pd.read_csv(input_path)
    
    # --- Step 1: Data Collection (raw count) ---
    global_statistic['Data collection'] += len(df)
    
    # --- Step 1b: Remove hyperlinks (needed for accurate word counts) ---
    df['body'] = df['body'].apply(remove_hyperlinks)
    
    # --- Step 2: Drop duplicates ---
    df = df.drop_duplicates(subset=['body'])
    word_counts = df['body'].apply(count_words)
    long_tweets = df[word_counts > 5]
    short_tweets = df[word_counts <= 5]
    long_tweets = long_tweets.drop_duplicates(subset=['body'])
    df = pd.concat([long_tweets, short_tweets], ignore_index=True)
    global_statistic['Drop duplicates'] += len(df)
    
    # Step 3: Filter by number of cashtags & hashtags 
    df['cashtags'] = df['body'].apply(count_cashtags)
    df['hashtags'] = df['body'].apply(count_hashtags)
    df = df[(df['cashtags'] < 5) & (df['hashtags'] < 8)]
    global_statistic['Filter number of cashtags & hashtags'] += len(df)
    
    # Step 4: Remove spam by ratios 
    df['mentions'] = df['body'].apply(count_mentions)
    df['words'] = df['body'].apply(count_words)
    safe = df['words'] > 0
    df = df[
        ~safe |
        (
            (df['cashtags'] / df['words'].clip(lower=1) <= 0.5) &
            (df['hashtags'] / df['words'].clip(lower=1) <= 0.5) &
            (df['mentions'] / df['words'].clip(lower=1) <= 0.5)
        )
    ]
    global_statistic['Remove spam by ratios'] += len(df)
    
    #  Step 5: Remove cryptocurrency posts 
    df['crypto_count'] = df['body'].apply(count_crypto_keywords)
    df = df[df['crypto_count'] <= 2]
    global_statistic['Remove cryptocurrency posts'] += len(df)
    
    # Drop helper columns
    df = df.drop(columns=['cashtags', 'hashtags', 'mentions', 'words', 'crypto_count'])
    
    # --- Step 6: Filter non-English tweets ---
    df = df[df['body'].apply(is_english)]
    global_statistic['Filter non-English'] += len(df)
    
    # --- Step 7: Clean text for model + filter short ---
    df['cleaned_body'] = df['body'].apply(lambda text: clean_for_model(text, ticker=ticker))
    df = df.dropna(subset=['cleaned_body'])
    global_statistic['Clean text for model'] += len(df)
    
    # --- Sentiment Analysis ---
    print(f"Analyzing sentiment for {len(df)} tweets...")
    sentiments = []
    for text in tqdm(df['cleaned_body'], desc="Progress"):
        sentiments.append(get_sentiment(text))
    df['sentiment'] = sentiments
    df = df.dropna(subset=['sentiment'])
    
    # --- Aggregate to daily average and save ---
    df['date'] = pd.to_datetime(df['post_date'], format='ISO8601').dt.date
    daily = df.groupby('date')['sentiment'].mean().reset_index()
    daily.columns = ['date', 'avg_sentiment']
    daily = daily.sort_values('date').reset_index(drop=True)
    
    out_path = OUTPUT_DIR / f"{ticker}_tweets_sentiment_daily.csv"
    daily.to_csv(out_path, index=False)
    
    print(f"Done: {len(df):,} tweets -> {len(daily)} daily rows saved")

print(f"\nAll files processed. Output: {OUTPUT_DIR}")


Processing tweets_AAPL.csv
Analyzing sentiment for 13230 tweets...


Progress: 100%|██████████| 13230/13230 [01:13<00:00, 180.30it/s]


Done: 13,230 tweets -> 715 daily rows saved

Processing tweets_AMZN.csv
Analyzing sentiment for 13609 tweets...


Progress: 100%|██████████| 13609/13609 [01:10<00:00, 193.95it/s]


Done: 13,609 tweets -> 724 daily rows saved

Processing tweets_AVGO.csv
Analyzing sentiment for 10617 tweets...


...

Done: 7,328 tweets -> 707 daily rows saved

Processing tweets_WMT.csv
Analyzing sentiment for 7959 tweets...


Progress: 100%|██████████| 7959/7959 [00:39<00:00, 202.06it/s]


Done: 7,959 tweets -> 687 daily rows saved

Processing tweets_XOM.csv
Analyzing sentiment for 8427 tweets...


Progress: 100%|██████████| 8427/8427 [00:42<00:00, 196.14it/s]

Done: 8,427 tweets -> 731 daily rows saved

All files processed. Output: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\Processed_Data\tweets_sentiment_daily


In [9]:
# Build summary table (Wilksch & Abramova, 2023 style)
steps = list(global_statistic.keys())
counts = list(global_statistic.values())
deltas = [''] + [f'{counts[i] - counts[i-1]:+,}' for i in range(1, len(counts))]

summary_table = pd.DataFrame({
    'Step': steps,
    'n tweets after step': [f'{c:,}' for c in counts],
    'Δn': deltas,
})

print("Sample size during data cleaning stages:")
display(summary_table)

Sample size during data cleaning stages:


,Step,n tweets after step,Δn
0,Data collection,"361,053",
1,Drop duplicates,"333,320","-27,733"
2,Filter number of cashtags & hashtags,"253,225","-80,095"
3,Remove spam by ratios,"249,146","-4,079"
4,Remove cryptocurrency posts,"249,096",-50
5,Filter non-English,"210,291","-38,805"
6,Clean text for model,"209,581",-710


## 9. Summary

**Pipeline:** `Raw_Data/Tweets/` → clean, filter, sentiment → `Processed_Data/tweets_sentiment_daily/`

**Output CSV Format (`TICKER_tweets_sentiment_daily.csv`):**
- `date`: Date
- `avg_sentiment`: Mean FinRoBERTa sentiment across all tweets for that day [-1, 1]